# Pipeline demo: detection, tracking, and pitch calibration

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

This notebook exercises the core of the system built for the thesis:

1. Clone the `pitchvision` package and install its dependencies.
2. Mount Google Drive and locate footage in the `BuildingAction` / `Goals` / `SetPieces` folders.
3. Sanity-check YOLOv8 detection (players + ball) on a single frame.
4. Calibrate the pitch homography automatically from detected pitch keypoints (with a manual fallback).
5. Run the full detection + ByteTrack tracking + homography pipeline over a clip.
6. Save the resulting per-frame pitch-coordinate tracks to Drive for the phase-specific analyses (goal-scoring opportunity, build-up, set pieces) built in later notebooks.

## 1. Clone the repository and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/football-spatial-analysis-thesis-lx13x7"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -e .

## 2. Mount Google Drive and locate footage

In [ ]:
from pitchvision import DriveConfig, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive")
print(drive_cfg.building_action_path)
print(drive_cfg.goals_path)
print(drive_cfg.set_pieces_path)

In [ ]:
from pitchvision import list_videos

goal_videos = list_videos(drive_cfg.goals_path)
print(f"Found {len(goal_videos)} videos in Goals/")

sample_video = goal_videos[0]
sample_video

## 3. Detection sanity check

Runs a COCO-pretrained YOLOv8 model on the first frame, restricted to the `person` (players/referees) and `sports ball` classes.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from pitchvision import PlayerBallDetector, VideoFrames

frames = VideoFrames(sample_video)
first_frame = frames.read_frame(0)

detector = PlayerBallDetector(weights="yolov8n.pt", confidence=0.3)
detections = detector.detect(first_frame)
print(f"{len(detections)} detections in frame 0")

vis = first_frame.copy()
for det in detections:
    x1, y1, x2, y2 = det.xyxy.astype(int)
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(vis, det.class_name, (x1, max(y1 - 5, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLOv8 detections (person + sports ball)")
plt.show()

## 4. Pitch calibration

Two options, in order of preference:

**4A. Automatic (default).** A pretrained YOLOv8-pose model ([`football-pitch-detection.pt`](https://github.com/roboflow/sports), trained to localise 32 standard pitch keypoints - corners, box corners, centre-circle tangents, ...) detects the keypoints directly in the frame; the confidently-detected ones are matched against their known metric positions and fed straight into `PitchCalibrator.from_point_pairs`. No manual clicking, and it works per-frame so it can also handle camera pans/cuts if you re-run it per shot.

**4B. Manual fallback.** If 4A can't find enough confident keypoints for a given clip (heavy occlusion, an unusual crop, unusual lighting), fall back to reading pixel coordinates off the frame by hand.

In [ ]:
from pitchvision import PitchKeypointDetector, download_pitch_keypoint_weights

# Cached on Drive so you don't re-download this every Colab session.
weights_path = download_pitch_keypoint_weights(
    "/content/drive/MyDrive/pitchvision_models/football-pitch-detection.pt"
)

keypoint_detector = PitchKeypointDetector(weights=weights_path, confidence=0.5)
calibrator = keypoint_detector.calibrate(first_frame)

xy, conf = keypoint_detector.detect(first_frame)
confident = conf >= keypoint_detector.confidence
print(f"{confident.sum()}/32 pitch keypoints detected with confidence >= {keypoint_detector.confidence}")

In [ ]:
vis = first_frame.copy()
for (x, y), c, is_confident in zip(xy, conf, confident):
    color = (0, 255, 0) if is_confident else (0, 0, 255)
    cv2.circle(vis, (int(x), int(y)), 6, color, -1)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Detected pitch keypoints (green = used for calibration, red = below confidence threshold)")
plt.show()

### 4B. Manual fallback (optional - skip if 4A worked)

Only use this if `keypoint_detector` above couldn't find enough confident keypoints for this clip. Hover over the image below to read exact pixel coordinates, fill in `landmark_pixels`, then run `calibrator = manual_calibrator` to use it instead of the automatic one for the rest of this notebook.

In [ ]:
import plotly.express as px

from pitchvision import PITCH_LANDMARKS_M

fig = px.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
fig.update_layout(title="Hover to read pixel coordinates for the landmarks below", height=700)
fig.show()

print("Available landmark names:", list(PITCH_LANDMARKS_M.keys()))

In [ ]:
import numpy as np

from pitchvision import PitchCalibrator

# EDIT THIS: replace every value with the pixel (x, y) you read off the hover
# tooltip above for that landmark, for THIS frame. Do not leave these defaults -
# they are placeholders and do not correspond to real points in your video.
landmark_pixels = {
    "left_penalty_top": (123, 456),
    "left_penalty_bottom": (110, 620),
    "left_six_yard_top": (250, 430),
    "centre_spot": (900, 300),
}

_PLACEHOLDER = {
    "left_penalty_top": (123, 456),
    "left_penalty_bottom": (110, 620),
    "left_six_yard_top": (250, 430),
    "centre_spot": (900, 300),
}
assert landmark_pixels != _PLACEHOLDER, (
    "landmark_pixels still holds the placeholder values - edit them with real "
    "pixel coordinates read off the hover tooltip in the cell above before continuing."
)

pixel_points = np.array(list(landmark_pixels.values()))
pitch_points = np.array([PITCH_LANDMARKS_M[name] for name in landmark_pixels])

manual_calibrator = PitchCalibrator.from_point_pairs(pixel_points, pitch_points)
error_m = manual_calibrator.reprojection_error(pixel_points, pitch_points)
print("Mean reprojection error (m):", error_m)
if error_m > 2.0:
    print(
        "WARNING: reprojection error is high (>2m). Double-check that each pixel "
        "coordinate actually matches its landmark name and that all points come "
        "from this frame - a single mismatched pair can produce a garbage homography."
    )

# Uncomment to actually use this fallback calibration for the rest of the notebook:
# calibrator = manual_calibrator

### Sanity-check the calibration

Project the frame's detected players onto the top-down pitch view; they should land inside the pitch outline in positions that roughly match the original frame.

In [ ]:
from pitchvision import PITCH_LENGTH_M, PITCH_WIDTH_M, draw_pitch, plot_positions

foot_points = np.array([d.foot_point for d in detections if d.class_name == "person"])
pitch_positions = calibrator.pixel_to_pitch(foot_points)

margin = 10.0  # metres of slack around the pitch outline
in_bounds = (
    (pitch_positions[:, 0] >= -margin) & (pitch_positions[:, 0] <= PITCH_LENGTH_M + margin) &
    (pitch_positions[:, 1] >= -margin) & (pitch_positions[:, 1] <= PITCH_WIDTH_M + margin)
)
if in_bounds.mean() < 0.8:
    print(
        f"WARNING: only {in_bounds.sum()}/{len(in_bounds)} projected players land "
        "near the pitch. The calibration is likely wrong - revisit cell 4A/4B above."
    )

ax = draw_pitch()
plot_positions(ax, pitch_positions, color="yellow", s=40, edgecolors="black")
plt.title("Detected players projected onto the pitch (frame 0)")
plt.show()

## 5. Full pipeline: detection + ByteTrack tracking + homography

`max_frames` caps the run while iterating on calibration; set it to `None` for the full clip.

In [ ]:
from pitchvision import PlayerTracker, TrackingPipeline

tracker = PlayerTracker(weights="yolov8n.pt", confidence=0.3)
pipeline = TrackingPipeline(tracker=tracker, calibrator=calibrator)

tracks_df = pipeline.run(sample_video, max_frames=250)
tracks_df.head()

In [ ]:
ax = draw_pitch()
for track_id, group in tracks_df[tracks_df["class_name"] == "person"].groupby("track_id"):
    ax.plot(group["pitch_x"], group["pitch_y"], linewidth=1, alpha=0.7)
plt.title("Player trajectories over the sampled frames")
plt.show()

## 6. Save tracks for downstream analysis

In [ ]:
output_dir = "/content/drive/MyDrive/pitchvision_outputs"
os.makedirs(output_dir, exist_ok=True)

out_path = os.path.join(output_dir, os.path.splitext(os.path.basename(sample_video))[0] + "_tracks.csv")
tracks_df.to_csv(out_path, index=False)
out_path

## Next steps

This notebook covers the shared foundation only. Not yet implemented, planned for later notebooks:

- **Team classification** (e.g. jersey-colour clustering) to split tracks into the two teams, needed by every downstream analysis.
- **Goal-scoring opportunity phase**: defensive compactness (inter-player distances) and space control (Voronoi diagrams).
- **Build-up phase**: Effective Playing Space via Convex Hull, and stretch/compactness via formation centroids.
- **Set pieces / breaks in play**: repeatability of positional structures, and static-to-dynamic transition timing after the restart.